In [0]:
import os

# ★ここにあなたのGitHubトークンを貼り付けてください★
github_token = ""

# トークンを環境変数に設定
os.environ['GITHUB_TOKEN'] = github_token

print("✅ GitHubトークン設定完了")
print(f"   トークン長: {len(github_token)}文字")
print(f"   先頭: {github_token[:8]}...")

In [0]:
import requests
import base64
import os

os.chdir('/Workspace/Users/7716no70@gmail.com/naviko-lab')

print("🔧 GitHub API経由でWorkspace → GitHub同期開始\n")

# Step 1: トークン取得
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')
if not GITHUB_TOKEN:
    print("❌ GitHubトークンが設定されていません")
    print("   手順2を先に実行してください")
else:
    print(f"✅ トークン取得成功: {len(GITHUB_TOKEN)}文字")
    
    # GitHub API設定
    GITHUB_REPO = '7716no70-sudo/naviko-lab'
    GITHUB_API_URL = f'https://api.github.com/repos/{GITHUB_REPO}'
    
    # Step 2: naviko.py読み込み
    print("\n【Step 2: naviko.py読み込み】")
    with open('naviko.py', 'r', encoding='utf-8') as f:
        content = f.read()
    
    file_size = len(content.encode('utf-8'))
    print(f"✅ ファイル読み込み成功: {file_size:,} bytes")
    
    # Step 3: GitHub最新コミットSHA取得
    print("\n【Step 3: GitHub最新コミットSHA取得】")
    headers = {
        'Authorization': f'token {GITHUB_TOKEN}',
        'Accept': 'application/vnd.github.v3+json'
    }
    
    response = requests.get(
        f'{GITHUB_API_URL}/contents/naviko.py',
        headers=headers
    )
    
    if response.status_code == 200:
        current_file = response.json()
        current_sha = current_file['sha']
        print(f"✅ 現在のSHA取得成功: {current_sha[:8]}...")
    else:
        print(f"⚠️ SHA取得失敗: {response.status_code}")
        current_sha = None
    
    # Step 4: ファイルをBase64エンコード
    print("\n【Step 4: ファイルエンコード】")
    content_bytes = content.encode('utf-8')
    content_base64 = base64.b64encode(content_bytes).decode('utf-8')
    print(f"✅ エンコード完了: {len(content_base64)}文字")
    
    # Step 5: GitHub APIでファイル更新
    print("\n【Step 5: GitHub APIでファイル更新】")
    update_data = {
        'message': 'fix: Phase D-2-3完全完了 - ×ボタンwithdraw()対応+バックグラウンド維持',
        'content': content_base64,
        'branch': 'main'
    }
    
    if current_sha:
        update_data['sha'] = current_sha
    
    print("⚠️ GitHubへのアップロードを実行しますか？")
    print("   実行する場合は、次のセルで 'YES' と入力してください")

In [0]:
import requests
import os

os.chdir('/Workspace/Users/7716no70@gmail.com/naviko-lab')

print("🚀 最終同期実行開始\n")

# トークン取得
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')
GITHUB_REPO = '7716no70-sudo/naviko-lab'
GITHUB_API_URL = f'https://api.github.com/repos/{GITHUB_REPO}'

headers = {
    'Authorization': f'token {GITHUB_TOKEN}',
    'Accept': 'application/vnd.github.v3+json'
}

# 前のセルで準備したデータを使用
import base64

with open('naviko.py', 'r', encoding='utf-8') as f:
    content = f.read()

content_base64 = base64.b64encode(content.encode('utf-8')).decode('utf-8')

# SHA取得
response = requests.get(f'{GITHUB_API_URL}/contents/naviko.py', headers=headers)
current_sha = response.json()['sha'] if response.status_code == 200 else None

# GitHub更新
update_data = {
    'message': 'fix: Phase D-2-3完全完了 - ×ボタンwithdraw()対応+バックグラウンド維持',
    'content': content_base64,
    'branch': 'main'
}

if current_sha:
    update_data['sha'] = current_sha

response = requests.put(
    f'{GITHUB_API_URL}/contents/naviko.py',
    headers=headers,
    json=update_data
)

if response.status_code in [200, 201]:
    result = response.json()
    print("✅ GitHub更新成功")
    print(f"   - コミットSHA: {result['commit']['sha'][:8]}...")
    print(f"   - コミットメッセージ: {result['commit']['message']}")
    print("\n🎉 Workspace → GitHub同期完了！")
else:
    print(f"❌ GitHub更新エラー: {response.status_code}")
    print(f"   エラー内容: {response.text[:200]}")
    